# 02 — Data Cleaning
## Goal
Turn raw values into trustworthy values without hiding problems.

### Key rules
- Preserve the raw dataset.
- Strip whitespace from categorical text.
- Convert numeric columns explicitly.
- Handle missing values through a modeling pipeline rather than manually filling everything with arbitrary constants.
- Correct the known GPS sign issue using the India-domain constraint, and document it.
- Keep `Time_Order_picked` out of the predictive feature set.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_path = "../data/Zomato Dataset.csv"
df = pd.read_csv(raw_path)

# Work on a copy so the raw file is never changed.
clean = df.copy()

# Standardize whitespace in text columns.
text_cols = clean.select_dtypes(include="object").columns
for col in text_cols:
    clean[col] = clean[col].astype("string").str.strip()

# Convert expected numeric columns.
numeric_cols = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "Vehicle_condition",
    "multiple_deliveries",
    "Time_taken (min)"
]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Parse order date.
clean["Order_Date"] = pd.to_datetime(
    clean["Order_Date"], format="%d-%m-%Y", errors="coerce"
)

print(clean.shape)
clean.head()

(45584, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,22:10,Fog,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,15:05,Stormy,High,1,Meal,motorcycle,1.0,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,17:40,Sandstorms,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,09:30,Sandstorms,Low,0,Buffet,motorcycle,0.0,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,20:05,Fog,Jam,1,Snack,scooter,1.0,No,Metropolitian,41


In [2]:
# The dataset is India-based, so negative restaurant coordinates are treated as
# sign-corruption rather than valid Indian coordinates.
# We do NOT touch the delivery coordinates because they are already positive.

gps_cols = [
    "Restaurant_latitude",
    "Restaurant_longitude"
]

negative_counts = (clean[gps_cols] < 0).sum()
negative_counts

Restaurant_latitude     431
Restaurant_longitude    162
dtype: int64

In [3]:
clean["Restaurant_latitude"] = clean["Restaurant_latitude"].abs()
clean["Restaurant_longitude"] = clean["Restaurant_longitude"].abs()

# Ratings should represent a 1–5 scale.
# Instead of deleting an otherwise usable order, invalid ratings become missing
# and are later imputed inside the ML pipeline.
clean.loc[~clean["Delivery_person_Ratings"].between(1, 5), "Delivery_person_Ratings"] = np.nan

# Impossible ages are also treated as missing.
clean.loc[~clean["Delivery_person_Age"].between(18, 70), "Delivery_person_Age"] = np.nan

# Vehicle condition and multiple deliveries are expected in the observed 0–3 range.
clean.loc[~clean["Vehicle_condition"].between(0, 3), "Vehicle_condition"] = np.nan
clean.loc[~clean["multiple_deliveries"].between(0, 3), "multiple_deliveries"] = np.nan

# Target must exist and be positive.
clean = clean.dropna(subset=["Time_taken (min)"])
clean = clean[clean["Time_taken (min)"] > 0]

# Remove exact duplicate rows.
clean = clean.drop_duplicates()

print("Clean shape:", clean.shape)
clean.isna().sum().sort_values(ascending=False).head(10)

Clean shape: (45584, 20)


Delivery_person_Ratings    1961
Delivery_person_Age        1892
Time_Orderd                1731
City                       1200
multiple_deliveries         993
Weather_conditions          616
Road_traffic_density        601
Festival                    228
ID                            0
Type_of_vehicle               0
dtype: int64

In [4]:
# Save cleaned raw-level data.
Path("../outputs/cleaned").mkdir(parents=True, exist_ok=True)
clean.to_csv("../outputs/cleaned/zomato_cleaned_raw_level.csv", index=False)

clean.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,22:10,Fog,Jam,2.0,Snack,motorcycle,3.0,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,15:05,Stormy,High,1.0,Meal,motorcycle,1.0,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,17:40,Sandstorms,Medium,1.0,Drinks,scooter,1.0,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,09:30,Sandstorms,Low,0.0,Buffet,motorcycle,0.0,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,20:05,Fog,Jam,1.0,Snack,scooter,1.0,No,Metropolitian,41


## Why are we not simply using `dropna()`?

Dropping every row with any missing value would throw away many usable observations.

Instead, the final ML pipeline will use:
- median imputation for numeric features
- most-frequent imputation for categorical features
- one-hot encoding for categorical variables
- scaling where required

This keeps preprocessing reproducible and prevents information leakage because these transformations are fitted only on training data.